# 1D 装箱 u120_00 —— 对偶/割推导总览（模式覆盖结构）

问题：$\min\{\sum_p x_p:\ \sum_p a_{ip}x_p=1,\ x_p\in\mathbb{Z}_+\}$，列=装箱模式
（$\sum s_i a_i\le150$）。最优 48（长度下界 48 = 48 箱解）；LP 松弛 47.265957。

## 01 直接建模 —— 无对偶
证明链：长度下界 48 + 划分池 MIP 48 箱解（每件恰一次、总负载校验）；CP-SAT K=48 直接搜索 UNKNOWN。

## 02 列生成 —— LP 对偶
RMP 对偶：$\max\sum_i\pi_i$ s.t. $\sum_i a_{ip}\pi_i\le 1\ \forall$模式（划分约束 ⇒ 对偶变量自由，
本实现用覆盖+划分 MIP 恢复）；$rc_p=1-\sum a_{ip}\pi_i$；定价 = 0/1 背包（容量 150）。

## 03 Benders —— 子问题对偶 → 割
SP(y) 对偶：$\max\sum_i\pi_i+\sum_p\sigma_p(M y_p)$，s.t. $\sum a_{ip}\pi_i+\sigma_p\le1$；
弱对偶 ⇒ 割 $\theta+\sum\lambda_p y_p\ge\sum\pi_i$。

## 04 拉格朗日 —— 乘子对偶 + 次梯度
$L(\lambda)=\sum_i\lambda_i+M\min(0,\ 1-v(\lambda))$；子问题=背包；$g_i=1-\sum_p a_{ip}x_p$；
对偶 = LP 松弛 47.266。

## 05 LBBD —— 逻辑割（1D 退化）
主问题 item→bin；子问题容量检查；超载箱 → 惰性容量割（最小超载核心）——
1D 下 LBBD 退化为惰性约束主问题。

## 07 Branch-and-Price —— 箱数分支
根 LP 47.35 分数 → 分支 Σx≤47（长度下界不可行）vs Σx≥48（节点划分池 MIP 恢复 48）。


In [1]:
# 数值验证：RMP 对偶 + 强对偶 + 定价值=1
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/bpp_falkenauer/scripts")
import bpp_core as bc
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

lp, patterns, sel, iters = bc.cg_min_rolls(max_iter=1500)
mm = mathopt.Model()
x = [mm.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"x{k}") for k in sel]
covers = []
for i in range(bc.N):
    covers.append(mm.add_linear_constraint(
        mathopt.fast_sum([x[k]*patterns[k][i] for k in range(len(sel))]) >= 1.0, name=f"c{i}"))
mm.minimize(mathopt.fast_sum(x))
res = mathopt.solve(mm, mathopt.SolverType.GLOP)
dv = res.dual_values()
pi = [max(0.0, dv[covers[i]]) for i in range(bc.N)]
dual_obj = sum(pi)
print(f"LP 目标 = {round(lp,6)} | 对偶目标 Σπ = {round(dual_obj,6)} | 强对偶: {abs(lp-dual_obj)<1e-6}")
v, isel = bc.knap_rebuild(pi)
print(f"定价背包最优值 v(π) = {round(v,6)} | 对偶约束 Σaπ<=1 满足: {v <= 1 + 1e-9}")
print(f"最负 rc = {round(1.0 - v, 8)}（≈0 ⇒ LP 最优）")


实例 u120_00 | 容量 150 | 件数 120 | 总长 7078 | 长度下界 48 | 文件自报最优 48
python 3.10.20 | ortools 9.15.6755


LP 目标 = 47.265957 | 对偶目标 Σπ = 47.265957 | 强对偶: True
定价背包最优值 v(π) = 1.0 | 对偶约束 Σaπ<=1 满足: True
最负 rc = -0.0（≈0 ⇒ LP 最优）
